# Paper support (notebook 15): Paired significance of the headline comparisons (Table `tab:sig`)

**What this is.** The reproducibility notebook for the paper's significance table. The
manuscript reports paired tests for the headline localization comparisons, but the
*inferential* statistics (Wilcoxon p, bootstrap CI, effect size $d_z$) were not part of
the released per-experiment summaries — only the point estimates were. This notebook
closes that gap: it recomputes the entire `tab:sig` table directly from the released raw
per-run tables, so every number in it is traceable.

**Inputs (all already in the repo, no model training):**
- `reports/tables/10b_drift_localization_raw.csv` — UNSW, per `family x model x seed x method x gt` (notebook 10b)
- `reports/tables/07_localization_raw.csv` — CICIDS, per `day x seed x method x gt x k` (notebook 07)
- `reports/tables/06_baseline_metrics.csv` — used to *derive* the post-drift days (cold-start collapse), not hard-code them

**Comparisons (focal vs reference), matching the paper:**

*Against the model-free drift truth ($\mathrm{GT_{in}}$):*
- KS vs domain-SHAP (UNSW, Wasserstein anchor)
- dual-IDS vs domain-SHAP (UNSW)
- dual vs aux (CICIDS, GT1, post-drift days)
- KS vs random (UNSW)

*Against the explanation-derived truth ($\mathrm{GT_{ex}}$, circular):*
- domain-SHAP vs KS (UNSW, oracle-SHAP anchor)

**Pre-registered method (D033).** Paired differences over `family x model x seed` on UNSW
($n{=}18$) and `day x seed` on CICIDS ($n{=}9$, post-drift only). For each comparison:
$\Delta=\overline{(\text{focal}-\text{ref})}$; a **seeded** $10^4$-resample bootstrap 95% CI
of the paired mean difference; the Wilcoxon signed-rank $p$; and $d_z=\Delta/\mathrm{sd}(\text{diff})$.
Significant at $p<0.05$. The bootstrap is seeded (`SEED=42`) so the CI is stable and
**canonical** — if the current `tab:sig` draft differs in the last CI digit, update the
table to match this notebook.

**Runtime.** Seconds (pandas + scipy only). This is a paper-support notebook, not an
experiment; it adds no new evidence, it only makes existing evidence reproducible. Lives in `99_paper_figures/`.

---

**v2 (three-dataset extension).** Adds the CSE-CIC-IDS2018 comparisons from
`25_cse2018_raw.csv` (notebook 25, Addendum A2) to the same framework:

*Against $\mathrm{GT_{in}}$ (Wasserstein):* KS vs domain-SHAP (2018, trees,
$n{=}24$ = 4 families × 2 models × 3 seeds, mirroring the UNSW tree-only
pairing); dual-IDS vs domain-SHAP (2018, all three models, $n{=}36$ — the P3
claim is all-models); KS vs random (2018, trees, $n{=}24$); **domain-SHAP
(MLP) vs KS (2018, $n{=}12$)** — a new row type: the P6 drift-marginality
boundary made inferential (positive $\Delta$ favours the MLP localizer).

*Against $\mathrm{GT_{ex}}$ (circular):* domain-SHAP vs KS (2018, trees,
$n{=}24$) — the circularity replication.

Cross-checks reconcile every recomputed 2018 $\Delta$ against
`phase6_25_cse2018_benchmark.json` exactly as the UNSW/CICIDS rows reconcile
against theirs. Pairing note carried from 10b/25's construction: KS and
random scores are identical across model classes within a (family, seed)
because the windows are redrawn from the same seeded generator per model;
the tree-only pooling therefore duplicates each KS pair once, identically to
the established UNSW n=18 convention (D033) — kept for consistency, and the
single-model MLP row is free of it.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys, json, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')
from src.utils.documentation import log_finding, add_journal_entry

ROOT = Path('/content/drive/MyDrive/phd_thesis')
TABLES = ROOT / 'reports' / 'tables'
PROCESSED = ROOT / 'data' / 'processed'
DECISIONS_LOG = ROOT / 'reports' / 'advisor_notes' / 'decisions_log.md'
SUMMARY_JSON = PROCESSED / 'sig_15_significance.json'

def log_decision(decision_id, decision, rationale, alternatives='',
                 context='99_paper_figures/15_significance_tests'):
    today = datetime.date.today().isoformat()
    if DECISIONS_LOG.exists() and f'## {decision_id} \u2014' in DECISIONS_LOG.read_text():
        print(f'[DECISION SKIPPED] {decision_id} already logged.'); return
    entry = (f'\n## {decision_id} \u2014 {today}\n**Context:** `{context}`\n\n'
             f'**Decision:** {decision}\n\n**Rationale:** {rationale}\n\n')
    if alternatives:
        entry += f'**Alternatives considered:** {alternatives}\n'
    with open(DECISIONS_LOG, 'a') as f:
        f.write(entry)
    print(f'[DECISION LOGGED] {decision_id}')

print('Setup complete.')

Setup complete.


In [3]:
log_decision(
    decision_id='D033',
    decision=(
        'Recompute the paper significance table (tab:sig) from the released raw per-run '
        'tables so every inferential statistic is reproducible. Pairing: family x model x '
        'seed on UNSW (10b raw, n=18) and day x seed on CICIDS (07 raw, GT1, k=10, '
        'post-drift days only, n=9). Per comparison report Delta = mean(focal-ref), a '
        'seeded 10^4-resample bootstrap 95% CI of the paired mean difference, the Wilcoxon '
        'signed-rank p, and d_z = Delta / sd(diff). Significant at p<0.05. Post-drift days '
        'are DERIVED from 06_baseline_metrics.csv (cold_start recall == 0), not hard-coded.'
    ),
    rationale=(
        'The point estimates in tab:sig already match the released means, but the p-values, '
        'CIs and effect sizes appeared in no released artifact, so a reviewer could not '
        'reproduce the table from the package. Computing them from the same raw tables that '
        'produced the means makes the central circularity claim (the only large, significant '
        'explanation advantage is against the explanation-derived truth) fully auditable. '
        'Deriving the post-drift day set from the baseline collapse keeps the CICIDS pairing '
        'principled and self-documenting.'
    ),
    alternatives=(
        'Hard-code the five comparisons numerically (rejected: not reproducible, defeats the '
        'purpose). Use a paired t-test instead of Wilcoxon (rejected: precision@k is discrete '
        'with small n; the signed-rank test is the pre-registered choice). Bootstrap the '
        'Wilcoxon p as well (unnecessary; the signed-rank p is exact enough at these n).'
    ),
)

[DECISION SKIPPED] D033 already logged.


In [4]:
log_decision(
    decision_id='D035',   # <-- CHECK decisions_log.md: D031/D032 collided earlier;
                          #     use the true next free ID for this and backfill 25/25b.
    decision=(
        'Extend the canonical significance notebook (D033) with the CSE-CIC-IDS2018 '
        'comparisons from 25_cse2018_raw.csv, same paired-statistics core and seeded '
        'bootstrap. New rows: KS vs domain-SHAP (2018, trees, n=24, GT_in); dual-IDS '
        'vs domain-SHAP (2018, all models, n=36, GT_in); KS vs random (2018, trees, '
        'n=24, GT_in); domain-SHAP(MLP) vs KS (2018, n=12, GT_in) as the inferential '
        'form of the P6 marginality boundary; domain-SHAP vs KS (2018, trees, n=24, '
        'GT_ex) as the circularity replication. Tree-only pooling for KS rows mirrors '
        'the UNSW convention (10b had RF+XGB only).'
    ),
    rationale=(
        'Table tab:sig must cover all three datasets after the Phase-6 replication; '
        'keeping the identical pairing and bootstrap makes the 2018 rows directly '
        'comparable to the existing ones, and the MLP row turns the drift-marginality '
        'boundary (notebooks 25/25b) from point estimates into a paired test.'
    ),
)

[DECISION SKIPPED] D035 already logged.


In [5]:
# ---- Configuration ----
SEED = 42
N_BOOT = 10_000
K_PRIMARY = 10
GT_IN_UNSW = 'wasserstein'          # model-free drift truth (10b)
GT_EX_UNSW = 'domain_oracle_shap'   # explanation-derived truth (10b)
CICIDS_GT = 'GT1'                    # primary GT in notebook 07 / phase1_07
rng = np.random.default_rng(SEED)
print(f'SEED={SEED}  N_BOOT={N_BOOT}  k={K_PRIMARY}')

SEED=42  N_BOOT=10000  k=10


In [6]:
# ---- Load raw tables; derive post-drift (collapsed) CICIDS days ----
unsw = pd.read_csv(TABLES / '10b_drift_localization_raw.csv')
cic  = pd.read_csv(TABLES / '07_localization_raw.csv')
base = pd.read_csv(TABLES / '06_baseline_metrics.csv')

# A day is 'post-drift' iff the frozen cold-start detector has collapsed on it (recall == 0).
collapsed_days = sorted(
    base.loc[(base['model'] == 'cold_start') & (base['recall'] == 0.0), 'day'].unique()
)
print('Post-drift (cold-start collapsed) days:', collapsed_days)

# UNSW paired frames (rows aligned by family x model x seed)
unsw_in = (unsw[unsw['gt'] == GT_IN_UNSW]
           .pivot_table(index=['family', 'model', 'seed'], columns='method', values='precision'))
unsw_ex = (unsw[unsw['gt'] == GT_EX_UNSW]
           .pivot_table(index=['family', 'model', 'seed'], columns='method', values='precision'))

# CICIDS paired frame (rows aligned by day x seed), post-drift days only
cic_in = (cic[(cic['gt'] == CICIDS_GT) & (cic['k'] == K_PRIMARY) & (cic['day'].isin(collapsed_days))]
          .pivot_table(index=['day', 'seed'], columns='method', values='precision'))

print(f'UNSW pairs: {len(unsw_in)} | CICIDS post-drift pairs: {len(cic_in)}')
assert len(unsw_in) == 18, 'expected 18 UNSW pairs (3 families x 2 models x 3 seeds)'
assert len(cic_in)  == 9,  'expected 9 CICIDS post-drift pairs (3 days x 3 seeds)'

# ---- v2: CSE-CIC-IDS2018 raw (notebook 25) ----
GT_IN_2018 = 'wasserstein'
GT_EX_2018 = 'domain_oracle_shap'
c18 = pd.read_csv(TABLES / '25_cse2018_raw.csv')
c18_in_trees = (c18[(c18['gt'] == GT_IN_2018) & (c18['model'].isin(['rf', 'xgb']))]
                .pivot_table(index=['family', 'model', 'seed'],
                             columns='method', values='precision'))
c18_in_all = (c18[c18['gt'] == GT_IN_2018]
              .pivot_table(index=['family', 'model', 'seed'],
                           columns='method', values='precision'))
c18_in_mlp = (c18[(c18['gt'] == GT_IN_2018) & (c18['model'] == 'mlp')]
              .pivot_table(index=['family', 'seed'],
                           columns='method', values='precision'))
c18_ex_trees = (c18[(c18['gt'] == GT_EX_2018) & (c18['model'].isin(['rf', 'xgb']))]
                .pivot_table(index=['family', 'model', 'seed'],
                             columns='method', values='precision'))
print(f'2018 pairs: trees {len(c18_in_trees)} | all {len(c18_in_all)} | mlp {len(c18_in_mlp)}')
assert len(c18_in_trees) == 24, 'expected 24 (4 families x 2 tree models x 3 seeds)'
assert len(c18_in_all) == 36,  'expected 36 (4 x 3 x 3)'
assert len(c18_in_mlp) == 12,  'expected 12 (4 x 3)'


Post-drift (cold-start collapsed) days: ['friday', 'thursday', 'wednesday']
UNSW pairs: 18 | CICIDS post-drift pairs: 9
2018 pairs: trees 24 | all 36 | mlp 12


In [7]:
# ---- Paired-statistics core ----
def paired_stats(focal, reference):
    """Delta, seeded bootstrap 95% CI, Wilcoxon signed-rank p, and d_z for paired arrays."""
    f = np.asarray(focal, dtype=float)
    r = np.asarray(reference, dtype=float)
    assert f.shape == r.shape and f.ndim == 1
    d = f - r
    n = len(d)
    delta = float(d.mean())
    sd = float(d.std(ddof=1))
    dz = float(delta / sd) if sd > 0 else float('nan')
    # seeded bootstrap CI of the paired mean difference
    idx = rng.integers(0, n, size=(N_BOOT, n))
    boot = d[idx].mean(axis=1)
    ci_lo, ci_hi = (float(x) for x in np.percentile(boot, [2.5, 97.5]))
    # Wilcoxon signed-rank (exact-ish for small n); undefined if all differences are zero
    try:
        p = float(wilcoxon(d).pvalue)
    except ValueError:
        p = float('nan')
    return dict(delta=delta, ci_lo=ci_lo, ci_hi=ci_hi, p=p, dz=dz, n=n)

COMPARISONS = [
    # (label, truth-bucket, focal series, reference series, circular?)
    ('KS vs.\\ domain-SHAP (UNSW)',       'GT_in', unsw_in['ks'],          unsw_in['domain_shap'], False),
    ('dual-IDS vs.\\ domain-SHAP (UNSW)', 'GT_in', unsw_in['dual_ids'],    unsw_in['domain_shap'], False),
    ('dual vs.\\ aux (CICIDS)',           'GT_in', cic_in['dual_gain'],    cic_in['aux_only'],     False),
    ('KS vs.\\ random (UNSW)',            'GT_in', unsw_in['ks'],          unsw_in['random'],      False),
    ('domain-SHAP vs.\\ KS (UNSW)',       'GT_ex', unsw_ex['domain_shap'], unsw_ex['ks'],          True),
    # ---- v2: CSE-CIC-IDS2018 (notebook 25, Addendum A2) ----
    ('KS vs.\\ domain-SHAP (2018, trees)',   'GT_in', c18_in_trees['ks'],        c18_in_trees['domain_shap'], False),
    ('dual-IDS vs.\\ domain-SHAP (2018)',    'GT_in', c18_in_all['dual_ids'],    c18_in_all['domain_shap'],   False),
    ('KS vs.\\ random (2018, trees)',        'GT_in', c18_in_trees['ks'],        c18_in_trees['random'],      False),
    ('domain-SHAP (MLP) vs.\\ KS (2018)',    'GT_in', c18_in_mlp['domain_shap'], c18_in_mlp['ks'],            False),
    ('domain-SHAP vs.\\ KS (2018, trees)',   'GT_ex', c18_ex_trees['domain_shap'], c18_ex_trees['ks'],        True),
]

rows = []
for label, bucket, focal, ref, circular in COMPARISONS:
    s = paired_stats(focal, ref)
    s.update(comparison=label, bucket=bucket, circular=circular,
             significant=bool(s['p'] < 0.05) if s['p'] == s['p'] else False)
    rows.append(s)

sig = pd.DataFrame(rows)[['comparison', 'bucket', 'circular', 'n',
                          'delta', 'ci_lo', 'ci_hi', 'p', 'dz', 'significant']]
sig

,comparison,bucket,circular,n,delta,ci_lo,ci_hi,p,dz,significant
0,KS vs.\ domain-SHAP (UNSW),GT_in,False,18,0.044444,-0.016667,0.111111,0.503471,0.312236,False
1,dual-IDS vs.\ domain-SHAP (UNSW),GT_in,False,18,0.005556,-0.066667,0.083333,0.968518,0.034168,False
2,dual vs.\ aux (CICIDS),GT_in,False,9,-0.044444,-0.088889,-0.011111,0.250000,-0.611775,False
3,KS vs.\ random (UNSW),GT_in,False,18,0.144444,0.088889,0.200000,0.001839,1.116674,True
4,domain-SHAP vs.\ KS (UNSW),GT_ex,True,18,0.227778,0.172222,0.283333,0.000275,1.855759,True
5,"KS vs.\ domain-SHAP (2018, trees)",GT_in,False,24,0.129167,0.070833,0.183333,0.000816,0.885322,True
6,dual-IDS vs.\ domain-SHAP (2018),GT_in,False,36,-0.030556,-0.075000,0.011111,0.223329,-0.226602,False
7,"KS vs.\ random (2018, trees)",GT_in,False,24,0.350000,0.308333,0.391667,0.000017,3.291890,True
8,domain-SHAP (MLP) vs.\ KS (2018),GT_in,False,12,0.125000,0.058333,0.191667,0.007812,0.970454,True
9,"domain-SHAP vs.\ KS (2018, trees)",GT_ex,True,24,0.220833,0.154167,0.291667,0.000188,1.248498,True


In [8]:
# ---- Cross-check: recomputed Deltas must match the released phase summaries ----
with open(PROCESSED / 'phase2_10b_drift_localization.json') as f:
    j10b = json.load(f)
with open(PROCESSED / 'phase1_07_localization.json') as f:
    j07 = json.load(f)

checks = {
    'KS vs.\\ domain-SHAP (UNSW)':  j10b['wasserstein_anchor']['ks_minus_domainshap'],
    'dual-IDS vs.\\ domain-SHAP (UNSW)': j10b['wasserstein_anchor']['dual_minus_domainshap'],
    'dual vs.\\ aux (CICIDS)':      j07['q007_margin_dual_minus_aux'],
}
print('Cross-check recomputed Delta vs released summary:')
ok = True
for label, expected in checks.items():
    got = float(sig.loc[sig['comparison'] == label, 'delta'].iloc[0])
    match = abs(got - expected) < 1e-9
    ok &= match
    print(f'  {label:36s} recomputed {got:+.4f} | summary {expected:+.4f} | {"OK" if match else "MISMATCH"}')
print(f'  GT_ex domain-SHAP vs KS Delta = '
      f'{float(sig.loc[sig.comparison.str.startswith("domain-SHAP"), "delta"].iloc[0]):+.4f} '
      f'(paper +0.228)')
# ---- v2: reconcile 2018 deltas against phase6_25 summary ----
with open(PROCESSED / 'phase6_25_cse2018_benchmark.json') as f:
    j25 = json.load(f)
bm = j25['by_model']
checks18 = {
    'KS vs.\\ domain-SHAP (2018, trees)':
        np.mean([bm['rf']['ks_minus_dom'], bm['xgb']['ks_minus_dom']]),
    'dual-IDS vs.\\ domain-SHAP (2018)':
        np.mean([bm[m]['dual_minus_dom'] for m in ('rf', 'xgb', 'mlp')]),
    'domain-SHAP (MLP) vs.\\ KS (2018)': -bm['mlp']['ks_minus_dom'],
}
print('\n2018 cross-check recomputed Delta vs phase6_25 summary:')
for label, expected in checks18.items():
    got = float(sig.loc[sig['comparison'] == label, 'delta'].iloc[0])
    match = abs(got - float(expected)) < 1e-9
    ok &= match
    print(f'  {label:40s} recomputed {got:+.4f} | summary {float(expected):+.4f} | {"OK" if match else "MISMATCH"}')

assert ok, 'Recomputed deltas disagree with released summaries — investigate before using.'
print('\nAll deltas (UNSW, CICIDS, 2018) reconcile with the released per-experiment summaries.')

Cross-check recomputed Delta vs released summary:
  KS vs.\ domain-SHAP (UNSW)           recomputed +0.0444 | summary +0.0444 | OK
  dual-IDS vs.\ domain-SHAP (UNSW)     recomputed +0.0056 | summary +0.0056 | OK
  dual vs.\ aux (CICIDS)               recomputed -0.0444 | summary -0.0444 | OK
  GT_ex domain-SHAP vs KS Delta = +0.2278 (paper +0.228)

2018 cross-check recomputed Delta vs phase6_25 summary:
  KS vs.\ domain-SHAP (2018, trees)        recomputed +0.1292 | summary +0.1292 | OK
  dual-IDS vs.\ domain-SHAP (2018)         recomputed -0.0306 | summary -0.0306 | OK
  domain-SHAP (MLP) vs.\ KS (2018)         recomputed +0.1250 | summary +0.1250 | OK

All deltas (UNSW, CICIDS, 2018) reconcile with the released per-experiment summaries.


In [9]:
# ---- Render the LaTeX body for tab:sig and save artifacts ----
def fmt_row(r):
    r"""Match the paper's tab:sig convention: \textbf label, $\mathbf{...}$ numeric cells."""
    bold = r['significant']
    def num(x):
        return ('\\mathbf{' + x + '}') if bold else x      # emboldened inside math mode
    pstr = '<0.001' if r['p'] < 0.001 else ('%.3f' % r['p']).rstrip('0').rstrip('.')
    delta = '$' + num('%+.3f' % r['delta']) + '$'
    ci = '$' + num('[%+.2f,%+.2f]' % (r['ci_lo'], r['ci_hi'])) + '$'
    p = '$' + num(pstr) + '$'
    dz = '$' + num('%+.2f' % r['dz']) + '$'
    label = ('\\textbf{' + r['comparison'] + '}') if bold else r['comparison']
    return label + ' & ' + delta + ' & ' + ci + ' & ' + p + ' & ' + dz + '\\\\'

lines = [r'\multicolumn{5}{l}{\itshape Against model-free drift truth ($\mathrm{GT_{in}}$)}\\']
for _, r in sig[sig['bucket'] == 'GT_in'].iterrows():
    lines.append(fmt_row(r))
lines.append(r'\midrule')
lines.append(r'\multicolumn{5}{l}{\itshape Against explanation-derived truth ($\mathrm{GT_{ex}}$, circular)}\\')
for _, r in sig[sig['bucket'] == 'GT_ex'].iterrows():
    lines.append(fmt_row(r))

latex_body = '\n'.join(lines)
print('=== paste into the tabular body of Table tab:sig ===\n')
print(latex_body)

(TABLES / '15_tabsig.tex').write_text(latex_body + '\n')
sig.to_csv(TABLES / '15_significance_tabsig.csv', index=False)

pair_dump = []
for label, bucket, focal, ref, _ in COMPARISONS:
    diff = (np.asarray(focal, float) - np.asarray(ref, float))
    for i, dval in enumerate(diff):
        pair_dump.append(dict(comparison=label, bucket=bucket, pair_index=i, diff=float(dval)))
pd.DataFrame(pair_dump).to_csv(TABLES / '15_significance_pairs.csv', index=False)
print('\nsaved: 15_tabsig.tex, 15_significance_tabsig.csv, 15_significance_pairs.csv')

=== paste into the tabular body of Table tab:sig ===

\multicolumn{5}{l}{\itshape Against model-free drift truth ($\mathrm{GT_{in}}$)}\\
KS vs.\ domain-SHAP (UNSW) & $+0.044$ & $[-0.02,+0.11]$ & $0.503$ & $+0.31$\\
dual-IDS vs.\ domain-SHAP (UNSW) & $+0.006$ & $[-0.07,+0.08]$ & $0.969$ & $+0.03$\\
dual vs.\ aux (CICIDS) & $-0.044$ & $[-0.09,-0.01]$ & $0.25$ & $-0.61$\\
\textbf{KS vs.\ random (UNSW)} & $\mathbf{+0.144}$ & $\mathbf{[+0.09,+0.20]}$ & $\mathbf{0.002}$ & $\mathbf{+1.12}$\\
\textbf{KS vs.\ domain-SHAP (2018, trees)} & $\mathbf{+0.129}$ & $\mathbf{[+0.07,+0.18]}$ & $\mathbf{<0.001}$ & $\mathbf{+0.89}$\\
dual-IDS vs.\ domain-SHAP (2018) & $-0.031$ & $[-0.08,+0.01]$ & $0.223$ & $-0.23$\\
\textbf{KS vs.\ random (2018, trees)} & $\mathbf{+0.350}$ & $\mathbf{[+0.31,+0.39]}$ & $\mathbf{<0.001}$ & $\mathbf{+3.29}$\\
\textbf{domain-SHAP (MLP) vs.\ KS (2018)} & $\mathbf{+0.125}$ & $\mathbf{[+0.06,+0.19]}$ & $\mathbf{0.008}$ & $\mathbf{+0.97}$\\
\midrule
\multicolumn{5}{l}{\itshape Aga

In [10]:
# ---- Save summary JSON + log finding ----
SUMMARY = {
    'generated': datetime.datetime.now().isoformat(timespec='seconds'),
    'purpose': 'reproduce paper Table tab:sig (paired significance) from released raw tables',
    'seed': SEED, 'n_boot': N_BOOT, 'k': K_PRIMARY,
    'pairing': {'unsw': 'family x model x seed (n=18)',
                'cicids': 'day x seed, post-drift only (n=9)',
                'cse2018_trees': 'family x model x seed (n=24)',
                'cse2018_all': 'family x model x seed (n=36)',
                'cse2018_mlp': 'family x seed (n=12)'},
    'post_drift_days_cicids': collapsed_days,
    'gt_in_unsw': GT_IN_UNSW, 'gt_ex_unsw': GT_EX_UNSW, 'cicids_gt': CICIDS_GT,
    'rows': sig.to_dict(orient='records'),
}
SUMMARY_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(SUMMARY_JSON, 'w') as f:
    json.dump(SUMMARY, f, indent=2, default=str)
print(f'Saved {SUMMARY_JSON}')

# v2: set the next free finding_id, fill numbers from the printed table, uncomment.
# log_finding(
#     finding_id='F0XX',
#     title='tab:sig extended to three datasets (2018 rows added)',
#     context='99_paper_figures/15_significance_tests (v2)',
#     what_we_did='Added the five 2018 comparisons to the canonical paired-significance '
#                 'framework (same seeded bootstrap, same pairing convention).',
#     what_we_found='<2018 rows: KS-vs-dom trees, dual-vs-dom, KS-vs-random, '
#                   'MLP-dom-vs-KS (P6), circularity GT_ex — Deltas/CIs/p/dz>',
#     why_it_matters='Table IV now covers all three datasets; the MLP marginality '
#                    'boundary and the third circularity replication are inferential.',
#     surprised='<yes/no>',
# )
# --- original F021 call (already logged in the v1 run) ---
# log_finding(
#     finding_id='F021',
#     title='Paired significance table (tab:sig) made reproducible from released raw tables',
#     context='99_paper_figures/15_significance_tests',
#     what_we_did=(
#         'Recomputed the five headline paired comparisons (Delta, seeded 10^4 bootstrap 95% CI, '
#         'Wilcoxon signed-rank p, d_z) directly from 10b/07 raw tables; post-drift CICIDS days '
#         'derived from the cold-start collapse in 06_baseline_metrics.csv. D033.'
#     ),
#     what_we_found=(
#         'Against GT_in no explanation method significantly exceeds KS (all p>=0.50) and the dual '
#         'model is indistinguishable from its component; KS beats random (p=0.002, d_z=1.12); the '
#         'only large significant explanation advantage is against the explanation-derived truth '
#         'GT_ex (Delta=+0.228, p<0.001, d_z=1.86). All Deltas reconcile with the released summaries.'
#     ),
#     why_it_matters=(
#         'tab:sig is load-bearing for the central circularity claim but its p/CI/d_z were not '
#         'previously reproducible from the package. This notebook makes the whole table auditable '
#         'and canonical (seeded bootstrap), closing the one reproducibility gap a reviewer could hit.'
#     ),
#     surprised='no',
# )
# print('F021 logged.')


Saved /content/drive/MyDrive/phd_thesis/data/processed/sig_15_significance.json


## What this settles, and the one edit to make

Running this notebook regenerates Table `tab:sig` end-to-end from the released raw tables.
Drop the printed body (also saved to `reports/tables/15_tabsig.tex`) into the `tabular` of
`tab:sig` in `paper.tex`, and add a one-line reproducibility pointer, e.g.:

> *Significance (Table `tab:sig`) is regenerated by `notebooks/99_paper_figures/15_significance_tests.ipynb` from the released raw tables.*

**Expected values** (these match the manuscript): KS vs domain-SHAP (UNSW) $\Delta{=}{+}0.044$,
$p{=}0.50$, $d_z{=}{+}0.31$; dual-IDS vs domain-SHAP $\Delta{=}{+}0.006$, $p{=}0.97$;
dual vs aux (CICIDS) $\Delta{=}{-}0.044$, $p{=}0.25$, $d_z{=}{-}0.61$;
**KS vs random** $\Delta{=}{+}0.144$, $p{=}0.002$, $d_z{=}{+}1.12$;
**domain-SHAP vs KS (GT$_\mathrm{ex}$)** $\Delta{=}{+}0.228$, $p{<}0.001$, $d_z{=}{+}1.86$.

The seeded bootstrap CIs may differ from the *current* draft in the last digit (the draft
used an unseeded run). Two rows to check and sync to this notebook's output if they differ:
**KS vs random** (draft `[+0.09,+0.21]`) and **dual vs aux** (draft `[-0.09,+0.00]`). Treat
this notebook as the canonical source for those CIs from now on.

---

**v2 expected values (point estimates from notebooks 25/25b; CIs/p/d_z are new):**
KS vs domain-SHAP (2018, trees) $\Delta{=}{+}0.129$ (mean of RF ${+}0.058$, XGB ${+}0.200$);
dual-IDS vs domain-SHAP (2018) $\Delta{=}{-}0.030$ (mean of ${+}0.017,{-}0.050,{-}0.058$);
domain-SHAP (MLP) vs KS (2018) $\Delta{=}{+}0.125$;
domain-SHAP vs KS (2018, GT$_\mathrm{ex}$, trees): expect a large positive $\Delta$
(tree circularity ${+}0.425/{+}0.433$). Paste the regenerated `15_tabsig.tex` body into
the paper's Table `tab:sig` and fill the ⟨…⟩ placeholders in
`paper_edits_three_datasets.md` §9 from the printed rows.